# Notatnik 08b — strojenie hiperparametrów LPPLS + ML (walk-forward)
Porównanie XGBoost, LightGBM, CatBoost, regresji logistycznej i drzewa decyzyjnego.
Dla każdego algorytmu trzy zestawy cech: Price-only, LPPLS-only, Price+LPPLS.
Cechy są obliczane wyłącznie z danych z dnia t lub wcześniejszych.
Przed każdą granicą kalibracyjną i testową usuwamy etykiety z przyszłości.
Domyślnie tylko spadek >=10% / 60 dni; RUN_ALL_NINE=True uruchamia 9 konfiguracji.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             average_precision_score, f1_score)

ROOT = Path('.')
OUT = ROOT / '08b_wyniki_tuning'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
TEST_YEARS = [2021, 2022, 2023, 2024]
RUN_ALL_NINE = True
N_SEARCH_ITER = 20  # na test: 3 lub 5; do finalnego eksperymentu: 20–40
TARGET_CONFIGS = ([(d, h) for h in (30, 45, 60) for d in (10, 15, 20)]
                  if RUN_ALL_NINE else [(10, 60)])
THRESHOLDS = np.round(np.arange(0.10, 0.91, 0.05), 2)
MODEL_NAMES = ['LogisticRegression', 'DecisionTree', 'XGBoost', 'LightGBM', 'CatBoost']
print('Konfiguracje:', TARGET_CONFIGS)
print('Warianty w Randomized Search:', N_SEARCH_ITER)

## 1. Wczytaj zbiór z notatnika 07 i pełne dopasowania LPPLS
Wymagane pliki:
- `wyniki_future_drop_07/lppls_future_drop_dataset.csv` (lub w bieżącym folderze),
- `lppls_rolling_results.csv` (wszystkie okna, nie tylko zaakceptowane).
Jeśli brakuje pierwszego pliku, wykonaj najpierw notatnik 07 na rzeczywistych cenach BTC.

In [ ]:
def find_file(candidates):
    for path in map(Path, candidates):
        if path.exists():
            return path
    raise FileNotFoundError('Nie znaleziono pliku. Sprawdź ścieżki: ' + ', '.join(map(str, candidates)))

file07 = find_file([
    ROOT / '07_wyniki_future_drop' / 'lppls_future_drop_dataset.csv',
    ROOT / 'lppls_future_drop_dataset.csv'
])
# Use one explicitly selected, complete source for all experiments.
filefits = ROOT / 'lppls_rolling_results.csv'
if not filefits.is_file():
    raise FileNotFoundError(f'Missing complete LPPLS fits: {filefits}')

base = pd.read_csv(file07)
base['date'] = pd.to_datetime(base['date'], utc=True).dt.tz_convert(None).dt.normalize()
base = base.sort_values('date').reset_index(drop=True)
for col in ['price', 'confidence', 'n_accepted', 'n_all']:
    base[col] = pd.to_numeric(base[col], errors='raise')
assert base.date.is_unique and base.date.is_monotonic_increasing
assert base.price.notna().all() and base.price.gt(0).all()
assert base.confidence.between(0, 1).all()
assert base.date.diff().dropna().eq(pd.Timedelta(days=1)).all(), 'Brakuje dni w zbiorze 07'

for drop, horizon in TARGET_CONFIGS:
    y_col, e_col = f'drop_{drop}_h{horizon}', f'label_end_h{horizon}'
    assert y_col in base.columns and e_col in base.columns, f'Brak {y_col} lub {e_col}'
    base[y_col] = pd.to_numeric(base[y_col], errors='coerce')
    base[e_col] = pd.to_datetime(base[e_col], errors='coerce')
    valid = base[e_col].notna()
    assert (base.loc[valid, e_col] == base.loc[valid, 'date'] + pd.Timedelta(days=horizon)).all()
    assert base[y_col].isna().equals(~valid)

fits = pd.read_csv(filefits, parse_dates=['win_end'])
print('Dane z 07:', base.shape, '| Wszystkie okna:', fits.shape)
# This count is specific to the frozen BTC 2015–2024 run.
assert len(fits) == 107_240, f'Expected 107240 fits, found {len(fits)}'


## 2. Cechy cenowe: wyłącznie przeszłość i dzień prognozy
Nie przekazujemy modelom bieżącej ceny bezwzględnej, etykiet literaturowych
ani kolumn future_drop. Retrospektywne etykiety służą wyłącznie do oceny.

In [ ]:
data = base.copy()
price = data['price']
log_return = np.log(price / price.shift(1))

data['ret_1d'] = log_return
for n in (7, 14, 30, 60):
    data[f'momentum_{n}d'] = np.log(price / price.shift(n))
for n in (7, 30, 60):
    data[f'volatility_{n}d'] = log_return.rolling(n, min_periods=n).std()
for n in (30, 90):
    data[f'drawdown_{n}d'] = price / price.rolling(n, min_periods=n).max() - 1

PRICE_FEATURES = ([ 'ret_1d' ]
    + [f'momentum_{n}d' for n in (7, 14, 30, 60)]
    + [f'volatility_{n}d' for n in (7, 30, 60)]
    + [f'drawdown_{n}d' for n in (30, 90)])

## 3. Cechy LPPLS na poziomie dnia
Stosujemy te same kryteria co wcześniej: 0<m<1, 6<=w<=13, B<0,
0<t_gap<=40 dni, R2>0.90. Agregujemy parametry zaakceptowanych okien,
bez bezwzględnego tc i A, które mogą identyfikować poziom cen / okres.
Dni bez zaakceptowanych okien ZOSTAJĄ w próbie; mediany wtedy są NaN
i zostaną uzupełnione tylko na podstawie danych treningowych.

In [ ]:
# Build exactly the same daily LPPLS features used by notebook 14.
import sys
sys.path.insert(0, str(ROOT.resolve() / 'src'))
from lppls_research import build_lppls_daily_features, TemporalAvailabilityValidator
from lppls_research.features import LPPLS_METRICS

TemporalAvailabilityValidator.check_calendar(data, require_consecutive_days=True)
TemporalAvailabilityValidator.check_windows(fits)
lppls_daily = build_lppls_daily_features(fits, calendar=data['date'])
TemporalAvailabilityValidator.check_matching_counts(
    data, lppls_daily, mapping={'n_all': 'lppls_n_all',
                               'n_accepted': 'lppls_n_accepted',
                               'confidence': 'lppls_ci'}
)
# Keep the older notebook's raw n_all; replace counters and fit summaries.
data = data.drop(columns=['confidence', 'n_accepted']).merge(
    lppls_daily.drop(columns='lppls_n_all'), on='date', how='left', validate='one_to_one'
)
LPPLS_FEATURES = ['lppls_ci', 'lppls_n_accepted'] + list(LPPLS_METRICS)
FEATURE_GROUPS = {
    'Price-only': PRICE_FEATURES,
    'LPPLS-only': LPPLS_FEATURES,
    'Price+LPPLS': PRICE_FEATURES + LPPLS_FEATURES,
}

data.to_csv(OUT / '01_cechy_dzienne.csv', index=False)
print('All fits:', int(lppls_daily.lppls_n_all.sum()))
print('Accepted fits:', int(lppls_daily.lppls_n_accepted.sum()))
print('Price features:', PRICE_FEATURES)
print('LPPLS features:', LPPLS_FEATURES)
display(data[['date','lppls_ci','lppls_n_accepted','lppls_tgap']].head())


## 4. Przestrzenie hiperparametrów i metryki
Ograniczone zakresy zapobiegają niepotrzebnemu przeszukiwaniu ogromnej przestrzeni.
Wszelkie parametry wybieramy bez zaglądania do roku testowego.

In [ ]:
import json
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import roc_auc_score, average_precision_score

SEARCH_SPACE = {
    'LogisticRegression': {
        'C': [0.01, 0.1, 0.3, 1.0, 3.0, 10.0],
        'class_weight': [None, 'balanced']
    },
    'DecisionTree': {
        'max_depth': [2, 3, 4, 6, 8],
        'min_samples_leaf': [15, 30, 60, 100],
        'min_samples_split': [10, 30, 60],
        'class_weight': [None, 'balanced']
    },
    'XGBoost': {
        'n_estimators': [100, 200, 350],
        'max_depth': [2, 3, 5],
        'learning_rate': [0.02, 0.05, 0.10],
        'subsample': [0.7, 0.9, 1.0],
        'colsample_bytree': [0.7, 1.0],
        'min_child_weight': [1, 5, 15],
        'reg_lambda': [1, 5, 10]
    },
    'LightGBM': {
        'n_estimators': [100, 200, 350],
        'learning_rate': [0.02, 0.05, 0.10],
        'num_leaves': [7, 15, 31],
        'min_child_samples': [20, 40, 80],
        'colsample_bytree': [0.7, 1.0],
        'subsample': [0.8, 1.0],
    },
    'CatBoost': {
        'iterations': [100, 200, 350],
        'depth': [3, 4, 6],
        'learning_rate': [0.02, 0.05, 0.10],
        'l2_leaf_reg': [3, 10, 30],
    },
}

def make_model(name, params):
    if name == 'LogisticRegression':
        return LogisticRegression(max_iter=2500, random_state=SEED, **params)
    if name == 'DecisionTree':
        return DecisionTreeClassifier(random_state=SEED, **params)
    if name == 'XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(random_state=SEED, n_jobs=2, tree_method='hist',
                             eval_metric='logloss', **params)
    if name == 'LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(random_state=SEED, n_jobs=2, verbosity=-1, **params)
    if name == 'CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(random_seed=SEED, thread_count=2,
                                  verbose=False, allow_writing_files=False, **params)
    raise ValueError(name)


def prepare_arrays(x_train, x_pred, name):
    imputer = SimpleImputer(strategy='median', add_indicator=True,
                            keep_empty_features=True)
    train_mat = imputer.fit_transform(x_train)
    pred_mat = imputer.transform(x_pred)
    if name == 'LogisticRegression':
        scaler = StandardScaler()
        train_mat = scaler.fit_transform(train_mat)
        pred_mat = scaler.transform(pred_mat)
    return train_mat, pred_mat


def proba_fit_predict(name, params, train, val, features, target):
    Xtr, Xva = prepare_arrays(train[features], val[features], name)
    model = make_model(name, params)
    model.fit(Xtr, train[target].to_numpy(dtype=int))
    return model.predict_proba(Xva)[:, 1]


def evaluate(y, pred, score):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    score = np.asarray(score, dtype=float)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(n=len(y), positives=int(tp+fn), TP=int(tp), FP=int(fp),
                FN=int(fn), TN=int(tn), alarm_days=int(pred.sum()),
                precision=(tp/(tp+fp) if tp+fp else np.nan),
                recall=(tp/(tp+fn) if tp+fn else np.nan),
                specificity=(tn/(tn+fp) if tn+fp else np.nan),
                f1=(2*tp/(2*tp+fp+fn) if (tp+fn) else np.nan),
                roc_auc=(roc_auc_score(y, score) if len(np.unique(y))==2 else np.nan),
                avg_precision=(average_precision_score(y, score) if (tp+fn) else np.nan))


def choose_threshold(y, prob):
    y = np.asarray(y, dtype=int)
    if len(np.unique(y)) < 2:
        return 0.5, 'fallback_one_class'
    scores=[]
    for t in THRESHOLDS:
        pred=(prob>=t).astype(int)
        f1=f1_score(y, pred, zero_division=0)
        scores.append((f1, -int(pred.sum()), float(t)))
    best=max(scores, key=lambda x: (x[0], x[1], x[2]))
    return best[2], 'F1_threshold_validation'

## 5. Dobór hiperparametrów przez dwa WEWNĘTRZNE podziały chronologiczne
Dla roku testowego Y:
- walidacje parametrów: Y-3 oraz Y-2
- walidacja progu: Y-1
- test: Y
Trening w każdej iteracji używa tylko etykiet znanych przed początkiem walidacji.
Wynik wyszukiwania to średni Average Precision z dostępnych foldów wewnętrznych.

In [ ]:
def search_params(sample, year, model_name, group_name, features, target, end_col):
    threshold_start = pd.Timestamp(f'{year-1}-01-01')
    candidates = list(ParameterSampler(
        SEARCH_SPACE[model_name], n_iter=N_SEARCH_ITER,
        random_state=SEED + year + MODEL_NAMES.index(model_name)
    ))
    trials=[]
    for idx, params in enumerate(candidates):
        scores=[]
        for val_year in (year-3, year-2):
            val_start=pd.Timestamp(f'{val_year}-01-01')
            val_stop=pd.Timestamp(f'{val_year+1}-01-01')
            train=sample.loc[(sample.date<val_start) & (sample[end_col]<val_start)]
            val=sample.loc[(sample.date>=val_start) & (sample.date<val_stop)
                           & (sample[end_col]<threshold_start)]
            if train.empty or val.empty:
                continue
            if train[target].nunique()<2 or val[target].nunique()<2:
                continue
            assert train[end_col].max() < val.date.min()
            pr=proba_fit_predict(model_name, params, train, val, features, target)
            scores.append(average_precision_score(val[target].to_numpy(dtype=int),pr))
        trials.append(dict(
            year=year, model=model_name, feature_group=group_name,
            trial=idx+1, mean_AP=float(np.mean(scores)) if scores else np.nan,
            n_inner_folds=len(scores), params=json.dumps(params, sort_keys=True),
        ))
    valid=[t for t in trials if np.isfinite(t['mean_AP'])]
    if not valid:
        return None, trials
    best=max(valid, key=lambda r:(r['mean_AP'],r['n_inner_folds']))
    best['selected']=True
    for r in trials:
        r.setdefault('selected',False)
    return json.loads(best['params']), trials

## 6. Właściwy eksperyment: parametry -> próg -> test
Aby nie zmieniać kalibracji po wyborze progu, testujemy dokładnie model
wytrenowany przed rokiem walidacji progu; nie dokonujemy refitu na roku progu.
To konserwatywne rozwiązanie, które pozwala porównać modele bez ryzyka
przesunięcia rozkładu prawdopodobieństw po ponownym treningu.

In [ ]:
search_rows=[]; fold_rows=[]; prediction_rows=[]; selected_rows=[]
for drop,horizon in TARGET_CONFIGS:
    target=f'drop_{drop}_h{horizon}'
    end_col=f'label_end_h{horizon}'
    sample=data.loc[data[target].notna()].copy()
    sample[end_col]=pd.to_datetime(sample[end_col]); sample[target]=sample[target].astype(int)
    for year in TEST_YEARS:
        threshold_start=pd.Timestamp(f'{year-1}-01-01')
        test_start=pd.Timestamp(f'{year}-01-01')
        test_stop=pd.Timestamp(f'{year+1}-01-01')
        train_for_threshold=sample.loc[(sample.date<threshold_start)
                                      & (sample[end_col]<threshold_start)]
        threshold_val=sample.loc[(sample.date>=threshold_start)
                                  & (sample.date<test_start)
                                  & (sample[end_col]<test_start)]
        test=sample.loc[(sample.date>=test_start)&(sample.date<test_stop)]
        if any(d.empty for d in (train_for_threshold, threshold_val, test)):
            warnings.warn(f'Pusty okres dla {drop}/{horizon}, rok {year}'); continue
        if train_for_threshold[target].nunique()<2:
            warnings.warn(f'Brak dwóch klas w treningu {drop}/{horizon}, rok {year}'); continue
        TemporalAvailabilityValidator.check_split(
            train_for_threshold, threshold_val, test, label_end_col=end_col
        )
        print(f'Test {year} | target {drop}%/{horizon}d | train {len(train_for_threshold)}, threshold-val {len(threshold_val)}, test {len(test)}')
        y_test=test[target].to_numpy(dtype=int)

        baseline={
            'Always-positive': (np.ones(len(test), dtype=int), np.ones(len(test))),
            'CI > 0':((test.lppls_ci.to_numpy()>0).astype(int),test.lppls_ci.to_numpy()),
            'CI > 0.05':((test.lppls_ci.to_numpy()>0.05).astype(int),test.lppls_ci.to_numpy())
        }
        for name,(pred,score) in baseline.items():
            fold_rows.append(dict(year=year, drop_pct=drop,horizon=horizon,
                model=name, feature_group='Baseline', selected_threshold=np.nan,
                params='{}',**evaluate(y_test,pred,score)))
            prediction_rows.append(pd.DataFrame(dict(date=test.date.to_numpy(),year=year,
                drop_pct=drop,horizon=horizon, model=name,feature_group='Baseline',
                y_true=y_test, y_pred=pred, score=score)))

        for group, features in FEATURE_GROUPS.items():
            for name in MODEL_NAMES:
                params, trials=search_params(sample,year,name,group,features,target,end_col)
                for r in trials:
                    search_rows.append(dict(drop_pct=drop,horizon=horizon,**r))
                if params is None:
                    warnings.warn(f'Brak poprawnej wewnętrznej walidacji dla {year}/{name}/{group}')
                    continue
                val_score=proba_fit_predict(name,params,train_for_threshold,threshold_val,features,target)
                threshold,rule=choose_threshold(threshold_val[target].to_numpy(dtype=int),val_score)
                test_score=proba_fit_predict(name,params,train_for_threshold,test,features,target)
                pred=(test_score>=threshold).astype(int)
                params_json=json.dumps(params,sort_keys=True)
                selected_rows.append(dict(year=year,drop_pct=drop,horizon=horizon,
                    model=name,feature_group=group, params=params_json,
                    threshold=threshold,threshold_rule=rule))
                fold_rows.append(dict(year=year,drop_pct=drop,horizon=horizon,
                    model=name,feature_group=group,selected_threshold=threshold,
                    params=params_json,**evaluate(y_test,pred,test_score)))
                prediction_rows.append(pd.DataFrame(dict(date=test.date.to_numpy(),year=year,
                    drop_pct=drop,horizon=horizon,model=name,feature_group=group,
                    y_true=y_test,y_pred=pred,score=test_score)))

folds=pd.DataFrame(fold_rows)
preds=pd.concat(prediction_rows,ignore_index=True)
search_results=pd.DataFrame(search_rows)
selected=pd.DataFrame(selected_rows)
search_results.to_csv(OUT/'08b_01_wyniki_search.csv', index=False)
selected.to_csv(OUT/'08b_02_wybrane_hiperparametry.csv', index=False)
folds.to_csv(OUT/'08b_03_wyniki_roczne.csv', index=False)
preds.to_csv(OUT/'08b_04_predykcje_OOS.csv', index=False)
print('Zakończono strojenie. Liczba eksperymentów:',len(folds))
display(selected.head(10))

## 7. Łączne wyniki OOS + efekt LPPLS dla danego algorytmu
Nie wolno uśredniać ROC-AUC z lat; liczymy na połączonych prognozach OOS.

In [ ]:
rows=[]
for (drop,horizon,name,group), part in preds.groupby(
    ['drop_pct','horizon','model','feature_group'],sort=False
):
    rows.append(dict(drop_pct=drop,horizon=horizon,model=name,feature_group=group,
                     **evaluate(part.y_true,part.y_pred,part.score)))
summary=pd.DataFrame(rows).sort_values(['drop_pct','horizon','feature_group','model'])
summary.to_csv(OUT/'08b_05_wyniki_laczne_OOS.csv',index=False)
for drop,horizon in TARGET_CONFIGS:
    s=summary.loc[(summary.drop_pct==drop)&(summary.horizon==horizon)]
    display(s[['model','feature_group','precision','recall','f1','specificity',
               'roc_auc','avg_precision','alarm_days','FP']].round(3))
    ml=s.loc[s.feature_group!='Baseline']
    pr=ml.loc[ml.feature_group=='Price-only'].set_index('model')
    hy=ml.loc[ml.feature_group=='Price+LPPLS'].set_index('model')
    delta=pd.DataFrame({'delta_F1':hy.f1-pr.f1,
        'delta_AvgPrecision':hy.avg_precision-pr.avg_precision,
        'delta_ROC_AUC':hy.roc_auc-pr.roc_auc}).reset_index()
    display(delta.round(3))
    delta.to_csv(OUT/f'08b_06_wartosc_dodana_LPPLS_{drop}_{horizon}.csv',index=False)
    piv=ml.pivot(index='model',columns='feature_group',values='avg_precision')
    piv=piv.reindex(MODEL_NAMES)
    ax=piv.plot.bar(figsize=(10,5),rot=20)
    ax.set_title(f'Average Precision — BTC spadek >={drop}% / {horizon} dni (OOS)')
    ax.set_ylabel('Average Precision')
    ax.grid(axis='y',alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUT/f'08b_07_AP_modele_{drop}_{horizon}.png',dpi=180)
    plt.show()

print('Wyniki:',OUT.resolve())